In [15]:
from collections.abc import Sequence
from typing import cast

import matplotlib
import matplotlib.pyplot as plt
import torch
import torchvision.transforms.functional as F
import torchvision
from PIL import Image
from torchvision.utils import make_grid

import sys,os
from pathlib import Path
sys.path.append(str(Path(os.getcwd()).resolve().parent.parent))

from shimmer.modules.global_workspace import (
    GlobalWorkspaceFusion,
)

from shimmer_metaworld import DEBUG_MODE, PROJECT_DIR,LOGGER
from shimmer_metaworld.config import load_config
from shimmer_metaworld.logging import get_pil_image, batch_to_device
from metaworld_dataset import (
    MetaworldDataModule,
    DomainDesc,
    get_default_domains,
)
from shimmer_metaworld.modules.domains import load_pretrained_domains

matplotlib.use("Agg")


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
import repsim

**Loading and pickling model variables**

In [16]:
import pickle
#clip -10,10 to remove weird values but the just mean stdv norm
act = {
    "gw_ckpt" : "3vr7mt9r",
    "ckpt_epoch" : "248"
}
act_low_con = {
    "gw_ckpt" : "m6ze7rnt",
    "ckpt_epoch" : "255"
}
current_model = "act"


if current_model == "act":
    gw_ckpt = act_low_con["gw_ckpt"]
    ckpt_epoch = act_low_con["ckpt_epoch"]
else:
    gw_ckpt = no_act["gw_ckpt"]
    ckpt_epoch = no_act["ckpt_epoch"]


def image_grid_from_v_tensor(
    samples: Sequence[torch.Tensor],
    _: int,
    ncols: int,
) -> Image:
    image = make_grid(samples[0], nrow=ncols, pad_value=1).detach()
    return F.to_pil_image(image)


debug_mode = DEBUG_MODE
extra_config_files = ["train_gw.yaml"]
argv = []

LOGGER.debug(f"Debug mode: {debug_mode}")

config = load_config(
    PROJECT_DIR / "shimmer_metaworld"/ "config_template",
    load_files=extra_config_files,
    debug_mode=debug_mode,
    log_config=False,
    argv=argv,
)
print(gw_ckpt)
#seed_everything(config.seed, workers=True)

domain_classes = get_default_domains(
    {domain.domain_type.kind.value for domain in config.domains}
)
print(config.domains)
domain_modules, gw_encoders, gw_decoders = load_pretrained_domains(
    config.domains,
    config.global_workspace.latent_dim,
    config.global_workspace.encoders.hidden_dim,
    config.global_workspace.encoders.n_layers,
    config.global_workspace.decoders.hidden_dim,
    config.global_workspace.decoders.n_layers,
    is_linear=config.global_workspace.linear_domains,
    bias=config.global_workspace.linear_domains_use_bias,
)



ckpt_path = f'/mnt/datashare/yelhelw/checkpoints/shimmer-meta-{gw_ckpt}/epoch={ckpt_epoch}.ckpt'

domain_module = GlobalWorkspaceFusion.load_from_checkpoint(ckpt_path, domain_mods=domain_modules,
    gw_encoders=gw_encoders,
    gw_decoders=gw_decoders)
domain_module.eval().freeze()

domain_module.to(device)





m6ze7rnt
[LoadedDomainConfig(checkpoint_path=PosixPath('/mnt/datashare/yelhelw/checkpoints/shimmer-meta-00jpyucb/epoch=115.ckpt'), domain_type=<DomainModuleVariant.v_latents: (<DomainType.v_latents: DomainDesc(base='v', kind='v_latents')>, 'default')>, args={}), LoadedDomainConfig(checkpoint_path=PosixPath('/mnt/datashare/yelhelw/checkpoints/pretrained/domain_act.ckpt'), domain_type=<DomainModuleVariant.act: (<DomainType.act: DomainDesc(base='act', kind='act')>, 'default')>, args={})]


GlobalWorkspaceFusion(
  (gw_mod): GWModule(
    (domain_mods): ModuleDict(
      (v_latents): VisualLatentDomainModule(
        (visual_module): VisualDomainModule(
          (vae): VAE(
            (encoder): RAEEncoder(
              (layers): Sequential(
                (0): Conv2d(3, 64, kernel_size=(4, 4), stride=(4, 4), bias=False)
                (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
                (2): ReLU()
                (3): Conv2d(64, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
                (4): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
                (5): ReLU()
                (6): Conv2d(128, 257, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
                (7): BatchNorm2d(257, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
                (8): ReLU()
                (9): Conv2d(257, 514, kernel_size=(4, 4), stride=(2, 

In [ ]:
print(domain_classes.keys())
for name, module in domain_module.gw_mod.gw_decoders.items():
    print(name,"->",module)




In [17]:
#Load data
domain_classes = get_default_domains(["v_latents","attr"])

print(config.domain_proportions)
data_module = MetaworldDataModule(
        '/mnt/datashare/yelhelw/complex_dataset_V2',
        domain_classes,
        config.domain_proportions,
        batch_size=config.training.batch_size,
        num_workers=config.training.num_workers,
        seed=config.seed,
        ood_seed=config.ood_seed,
        domain_args=config.domain_data_args,
    )

data_module_combin = MetaworldDataModule(
        '/mnt/datashare/yelhelw/combin_tasks',
        domain_classes,
        config.domain_proportions,
        batch_size=config.training.batch_size,
        num_workers=config.training.num_workers,
        seed=config.seed,
        ood_seed=config.ood_seed,
        domain_args=config.domain_data_args,
    )




{frozenset({'attr', 'v'}): 1.0, frozenset({'act', 'attr', 'v'}): 0.4}


**Start of analysis**

In [ ]:
import pickle

model = "act"
with open(f"models/{model}/domain_module.pkl","rb") as f:
    domain_module = pickle.load(f)
with open(f"models/{model}/data_module.pkl","rb") as f:
    data_module = pickle.load(f)
with open(f"models/{model}/domain_modules.pkl","rb") as f:
    domain_modules = pickle.load(f)


__Action to vision grid__ 

In [ ]:
train_samples = data_module.get_samples("train",32,offset=1)
train_samples=batch_to_device(train_samples,device)
import numpy as np
img_latents = np.load("/mnt/datashare/yelhelw/saved_latents/train/domain_v.npy")
act = domain_module.encode_domain(train_samples[frozenset(["act","attr"])]["act"],"act")
print(len(act))
#latent_img_uni = domain_module.encode_domain(img_latents, "v_latents")
gw_latent_v = domain_module.gw_mod.encode({"v_latents": torch.from_numpy(img_latents).to(device)})
gw_latent_act = domain_module.gw_mod.encode({"act":act})

In [ ]:
import itertools
import torch
import numpy as np
# Generate all 27 combinations of (-1, 0, 1)^3
#combos = list(itertools.product([-1,0,1,], repeat=4))
combos = []
index = 0

for x in np.arange(-1,1,.25):
    for z in np.arange(-1,1,.25):
        for y in np.arange(0,1.25,.25):
            combos.append([x,y,z,0])
            #print(combos[index])
            index +=1

#for x in np.arange(0,1,.1):
#    combos.append([0,x,0,0])

# Convert to torch tensor and reshape into 3×3×3×3
act = torch.tensor(combos).to(device)
#act[:,2]=0.6
print(index)

gw_latent_act = domain_module.gw_mod.encode({"act":act.float()})
gw_latent_v_fused = domain_module.gw_mod.fuse(gw_latent_act, {"act": torch.ones(gw_latent_act['act'].size(0)).to(device)})
decoded_latent_uni = domain_module.gw_mod.decode(gw_latent_v_fused)

decoded_images = domain_modules['v_latents'].decode_images(decoded_latent_uni['v_latents'])



__Object-dependent visualization__

In [ ]:
object = "wall"
object_images_path = Path(f"/home/yelhelw/metaworld_GW/Myworld/tasks/new_objects/{object}")

images = []
for x in range(1000):
    path = object_images_path/f"{x}.png"
    #path = self.image_path / f"{index}.png"
    with Image.open(path) as image:
        image = image.convert("RGB")
    tensor_img = torchvision.transforms.functional.pil_to_tensor(image).to(device).to(dtype=torch.float32) / 255
    images.append(tensor_img)
images = torch.stack(images)

latent_v_wall = domain_modules['v_latents'].visual_module.vae.encoder(images[:])[0]
print(latent_v_wall.shape)

In [ ]:
object_images_path = Path(f"/home/yelhelw/metaworld_GW/Myworld/tasks/new_objects/no_{object}")
import torchvision
images = []
for x in range(1000):
    path = object_images_path/f"{x}.png"
    #path = self.image_path / f"{index}.png"
    with Image.open(path) as image:
        image = image.convert("RGB")
    tensor_img = torchvision.transforms.functional.pil_to_tensor(image).to(device).to(dtype=torch.float32) / 255
    images.append(tensor_img)
images = torch.stack(images)

latent_v_no_wall = domain_modules['v_latents'].visual_module.vae.encoder(images[:])[0]
print(latent_v_no_wall.shape)

In [ ]:
gw_latent_v_wall = domain_module.gw_mod.encode({"v_latents":latent_v_wall})
gw_latent_v_no_wall = domain_module.gw_mod.encode({"v_latents":latent_v_no_wall})

In [ ]:
gw_latent_v_wall_fused = domain_module.gw_mod.fuse(gw_latent_v_wall, {"v_latents": torch.ones(gw_latent_v_wall['v_latents'].size(0)).to(device)})
gw_latent_v_no_wall_fused = domain_module.gw_mod.fuse(gw_latent_v_no_wall, {"v_latents": torch.ones(gw_latent_v_no_wall['v_latents'].size(0)).to(device)})

__Latent UMAP structures__

In [18]:
from collections.abc import Mapping
from shimmer.modules.selection import FixedSharedSelection
from matplotlib.colors import ListedColormap
from tqdm import tqdm


def to_device(data: torch.Tensor | Mapping[str, torch.Tensor] | list, device: str):
    """Put the data Tensor or list on the device (GPU or CPU)"""
    if isinstance(data, torch.Tensor):
        return data.to(device)
    elif isinstance(data, list):
        return [value.to(device) for value in data]
    elif isinstance(data, Mapping):
        return {name: to_device(value, device) for name, value in data.items()}
    else:
        raise TypeError(f"Unsupported type: {type(data)}")


In [19]:
import numpy as np

modalities = ["v"]
keys = frozenset({"v_latents","attr"})
fuse = False

np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fusion_mech = FixedSharedSelection()

data_module.prepare_data()
data_module.setup()

data_module.val_dataset = {keys :data_module.val_dataset[keys]}
dataloaders = {
    #"train": data_module.train_dataloader(shuffle=False, drop_last=False),
    "val" : data_module.val_dataloader(),
}

for split, dataloader in dataloaders.items():
    latents: list[np.ndarray] = []
    latents_dict : dict = {m: [] for m in modalities}
    vae_latents_dict : dict = {m: [] for m in modalities}
    print(f"Saving {split}.")
    for batch, _, _ in tqdm(iter(dataloader), total=len(dataloader)):
        data = to_device(batch[keys], device)
   
        latent = {}
        vae_latent = {}
        for modality in modalities:
            if modality == 'v':
                latent['v_latents'] = domain_modules['v_latents'].encode(data['v_latents'])
                vae_latents_dict['v'].append(latent['v_latents'].detach().cpu().numpy().copy())
            else:
                latent[modality] = domain_modules[modality].encode(data[modality])
                vae_latents_dict[modality].append(latent[modality].detach().cpu().numpy().copy())
        else:
            latent = domain_module.gw_mod.encode(latent)
            if fuse:
                selection_scores = fusion_mech(latent, latent)
                latent_fuse = domain_module.gw_mod.fuse(latent, selection_scores)
                latents.append(latent_fuse.detach().cpu().numpy().copy())
            
                latent['v'] = latent.pop('v_latents')
                #tanh is applied here to the modality vectors to make them correspond to the fused vector in latents
                #that applies an activation function at the end
                latents_dict = {m: latents_dict[m] + [torch.tanh(latent[m]).detach().cpu().numpy()] for m in latents_dict.keys() if m in latent}

            else:
                latent['v'] = latent.pop('v_latents')
                latents_dict = {m: latents_dict[m] + [latent[m].detach().cpu().numpy().copy()] for m in latents_dict.keys() if m in latent}

    if len(latents) > 0:
        latent_vectors = np.concatenate(latents, axis=0)
        shuffle_latent_vectors = latent_vectors.copy()
        np.random.shuffle(shuffle_latent_vectors)
        latents_dict = {m: np.concatenate(v, axis=0) for m, v in latents_dict.items()}
        vae_latents_dict = {m: np.concatenate(v, axis=0) for m, v in vae_latents_dict.items()}
    else:
        latents_dict = {m: np.concatenate(v, axis=0) for m, v in latents_dict.items()}
        vae_latents_dict = {m: np.concatenate(v, axis=0) for m, v in vae_latents_dict.items()}
        #latent_vectors = np.concatenate([latents_dict[m] for m in latents_dict.keys()], axis=0)
        #train only on vision latents to resemble Kuske : 
        latent_vectors = latents_dict['v']
        shuffle_latent_vectors = latent_vectors.copy()
        np.random.shuffle(shuffle_latent_vectors)
    
print(len(shuffle_latent_vectors))

Saving val.


100%|██████████| 98/98 [00:04<00:00, 21.25it/s]


100085


In [20]:
modalities = ["v"]
keys = frozenset({"v_latents","attr"})
fuse = False

np.random.seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fusion_mech = FixedSharedSelection()

data_module_combin.prepare_data()
data_module_combin.setup()

data_module_combin.val_dataset = {keys :data_module_combin.val_dataset[keys]}
dataloaders = {
    #"train": data_module_combin.train_dataloader(shuffle=False, drop_last=False),
    "val" : data_module_combin.val_dataloader(),
}

for split, dataloader in dataloaders.items():
    latents: list[np.ndarray] = []
    latents_dict_combin : dict = {m: [] for m in modalities}
    vae_latents_dict : dict = {m: [] for m in modalities}
    print(f"Saving {split}.")
    for batch, _, _ in tqdm(iter(dataloader), total=len(dataloader)):
        data = to_device(batch[keys], device)
   
        latent = {}
        vae_latent = {}
        for modality in modalities:
            if modality == 'v':
                latent['v_latents'] = domain_modules['v_latents'].encode(data['v_latents'])
                vae_latents_dict['v'].append(latent['v_latents'].detach().cpu().numpy().copy())
            else:
                latent[modality] = domain_modules[modality].encode(data[modality])
                vae_latents_dict[modality].append(latent[modality].detach().cpu().numpy().copy())
        else:
            latent = domain_module.gw_mod.encode(latent)
            if fuse:
                selection_scores = fusion_mech(latent, latent)
                latent_fuse = domain_module.gw_mod.fuse(latent, selection_scores)
                latents.append(latent_fuse.detach().cpu().numpy().copy())
            
                latent['v'] = latent.pop('v_latents')
                #tanh is applied here to the modality vectors to make them correspond to the fused vector in latents
                #that applies an activation function at the end
                latents_dict_combin = {m: latents_dict[m] + [torch.tanh(latent[m]).detach().cpu().numpy()] for m in latents_dict_combin.keys() if m in latent}

            else:
                latent['v'] = latent.pop('v_latents')
                latents_dict_combin = {m: latents_dict_combin[m] + [latent[m].detach().cpu().numpy().copy()] for m in latents_dict_combin.keys() if m in latent}

    if len(latents) > 0:
        latent_vectors = np.concatenate(latents, axis=0)
        shuffle_latent_vectors = latent_vectors.copy()
        np.random.shuffle(shuffle_latent_vectors)
        latents_dict = {m: np.concatenate(v, axis=0) for m, v in latents_dict.items()}
        vae_latents_dict = {m: np.concatenate(v, axis=0) for m, v in vae_latents_dict.items()}
    else:
        latents_dict_combin = {m: np.concatenate(v, axis=0) for m, v in latents_dict_combin.items()}
        #latent_vectors = np.concatenate([latents_dict[m] for m in latents_dict.keys()], axis=0)
        #train only on vision latents to resemble Kuske : 
        latent_vectors_combin = latents_dict_combin['v']
        shuffle_latent_vectors_combin = latent_vectors_combin.copy()
        np.random.shuffle(shuffle_latent_vectors_combin)
    
print(len(shuffle_latent_vectors_combin))

Saving val.


100%|██████████| 40/40 [00:02<00:00, 14.21it/s]

40009


<p align="center">
    Representation similarity VAE GW
</p>

In [ ]:

print(latents_dict['act'].shape)

dist_v = repsim.compare(
    torch.from_numpy(latent_vectors),
    torch.from_numpy(vae_latents_dict['v']),
    method='angular_cka',
)

dist_attr = repsim.compare(
    torch.from_numpy(latent_vectors),
    torch.from_numpy(vae_latents_dict['attr']),
    method='angular_cka',
)

dist_act = repsim.compare(
    torch.from_numpy(latent_vectors),
    torch.from_numpy(vae_latents_dict['act']),
    method='angular_cka',
)

print("dist_v:",dist_v)
print("dist_attr:",dist_attr)
print("dist_act:",dist_act)

np.save("repsim.npy",[dist_v.numpy(),dist_attr.numpy(),dist_act.numpy()])

<p align="center">
    UMAP parameters sweep
</p>

In [ ]:
from umap.umap_ import nearest_neighbors
import pickle



knn = nearest_neighbors(shuffle_latent_vectors,
                              n_neighbors=100,
                              metric="cosine",
                              metric_kwds=None,
                              angular=True,
                              random_state=None,
                             )

with open("knn_ball_action.pkl","wb") as f:
    pickle.dump(knn,f)

In [ ]:
import pickle
with open("vae_reducers.pkl", "rb") as f:
    vae_reducers = pickle.load(f)



In [ ]:
import umap

n_neighbors = [5, 25,50, 100]
min_dists = [0.1, 0.2, 0.5, 0.9]

embeddings = np.zeros((4, 4, 100047, 2))
for i, k in enumerate(n_neighbors):
    for j, dist in enumerate(min_dists):
        print(k,dist)
        reducer = umap.UMAP(n_neighbors=k,
                                                      min_dist=dist,
                                                      precomputed_knn=knn,random_state=42
                                                      ).fit(shuffle_latent_vectors)
        print("reducer done")
        embeddings[i, j] = reducer.transform(latent_vectors)

In [ ]:
import matplotlib.pyplot as plt

fig, axs = plt.subplots(4, 4, figsize=(20, 20))

for i, ax_row in enumerate(axs):
    for j, ax in enumerate(ax_row):
        for obj_label, color in color_map.items():
                mask = obj_code == obj_label
                ax.scatter(embeddings[i,j,mask, 0],
                embeddings[i,j,mask, 1],
                c=color,
                alpha=0.8,
                s=5,
                label=obj_label,
                )

        ax.set_xticks([])
        ax.set_yticks([])
        if i == 0:
            ax.set_title("min_dist = {}".format(min_dists[j]), size=15)
        if j == 0:
            ax.set_ylabel("n_neighbors = {}".format(n_neighbors[i]), size=15)
fig.suptitle("UMAP embedding of GW latents with grid of parameters", y=0.92, size=20)
plt.subplots_adjust(wspace=0.05, hspace=0.05)
plt.savefig("graphs/act_con0.1/UMAP_sweep.png")
plt.close()

In [21]:
import umap
reducer = umap.UMAP(n_neighbors=50,min_dist=0.5,metric="cosine",random_state=42,metric_kwds=None).fit(shuffle_latent_vectors)

/mnt/datashare/yelhelw/shimmer_env/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


<p align="center">
    Object centered Umap figures
</p>

In [ ]:
embedding_wall = reducer.transform(gw_latent_v_wall_fused.detach().cpu().numpy())
embedding_no_wall = reducer.transform(gw_latent_v_no_wall_fused.detach().cpu().numpy())

In [ ]:
fig = plt.figure()

plt.scatter(embedding_wall[:,0],embedding_wall[:,1],color="green",s=1,label="wall")
plt.scatter(embedding_no_wall[:,0],embedding_no_wall[:,1],color="red",s=1,label="no wall")
plt.legend()
plt.savefig(f'graphs/{current_model}/Umap_gw_{object}.png')


In [ ]:
from sklearn import svm
model = svm.SVC(kernel='linear')
from sklearn.utils import shuffle
import numpy as np

points = np.concatenate([gw_latent_v_wall_fused.detach().cpu().numpy(),gw_latent_v_no_wall_fused.detach().cpu().numpy()],axis=0)

labels = np.concatenate([np.ones(len(gw_latent_v_wall_fused)),np.zeros(len(gw_latent_v_no_wall_fused))])
points, labels = shuffle(points, labels, random_state=42)

model.fit(points,labels)

embedding_points = np.concatenate([embedding_wall,embedding_no_wall])



In [ ]:
#metrics for "representation structure" in 2 dimensional UMAP
from sklearn.metrics import davies_bouldin_score, silhouette_score
sc = silhouette_score(embedding_points, labels)
dvs = davies_bouldin_score(embedding_points, labels)

print(sc,dvs)
print(len(points))
class0 = embedding_points[labels==0]
class1 = embedding_points[labels==1]

centroid0 = class0.mean(axis=0)
centroid1 = class1.mean(axis=0)

distance = np.linalg.norm(centroid0 - centroid1)

np.save("umap_metrics_no_act.npy",{"silhouette_score" : sc, 'dbs' : dvs,'centroid':distance},allow_pickle=True)
print(distance)

In [ ]:
from scipy.stats import ttest_rel
metric_act = np.load("umap_metrics_wall_act.npy",allow_pickle=True).item()
metric_no_act = np.load("umap_metrics_no_act.npy",allow_pickle=True).item()

for key in metric_act.keys():
    print(metric_act[key], metric_no_act[key])

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

y_pred = model.predict(points)
print(accuracy_score(labels, y_pred))
print(precision_score(labels, y_pred))
print(recall_score(labels, y_pred))
print(f1_score(labels, y_pred))
print(confusion_matrix(labels, y_pred))


In [ ]:
embedding_wall_vae = reducer.transform(latent_v_wall.detach().cpu().numpy())
embedding_no_wall_vae = reducer.transform(latent_v_no_wall.detach().cpu().numpy())

In [ ]:
fig = plt.figure()
plt.scatter(embedding_wall_vae[:,0],embedding_wall_vae[:,1],color="green",label="wall")
plt.scatter(embedding_no_wall_vae[:,0],embedding_no_wall_vae[:,1],color="red",label="no_wall")
plt.legend()
plt.savefig('graphs/no_act/Umap_vae_wall_cosine.png')

<p align="center">
    General Embeddings
</p>

In [ ]:
labels = np.load('/mnt/datashare/yelhelw/complex_dataset_V2/actions_val.npy', mmap_mode="r")
labels_train = np.clip(np.load('/mnt/datashare/yelhelw/complex_dataset_V2/actions_train.npy', mmap_mode="r"), -10, 10)
all_labels = np.repeat(labels, 3, axis=0)
attributes = np.load('/mnt/datashare/yelhelw/complex_dataset_V2/attributes_val.npy', mmap_mode="r")
wall = attributes[:,9]
ball = attributes[:,6]
goal = attributes[:,-1]

print(ball)
modality_names = {0: 'Vision (v)', 1: 'Attributes (attr)', 2: 'Actions (act)'}


labels_norm = np.clip(labels.copy(), -1, 1)

#labels_norm = labels.copy()
action_mean = np.mean(labels_train,axis=0)
action_stdv = np.std(labels_train,axis=0)
#for x in range(4):
#    labels_norm[:,x] = (labels_norm[:,x]-action_mean[x])/ action_stdv[x]
#    labels_norm[:,x] = np.clip(labels_norm[:,x],-1,1)
x_disp = labels_norm[:, 0]
y_disp = labels_norm[:, 1]
z_disp = labels_norm[:, 2]
gripper = labels_norm[:, 3]

print("Transforming latent vectors...")
print(len(x_disp))
#embedding_all = reducer.transform(latent_vectors)
x_disp_full = np.copy(x_disp)
y_disp_full = np.copy(y_disp)
z_disp_full = np.copy(z_disp)
gripper_full = np.copy(gripper)
for img_latent in latent_vectors[len(x_disp):]:
    decoded_latent_uni = domain_module.gw_mod.decode(torch.from_numpy(img_latent).unsqueeze(0).to(device))
    decoded_act = np.clip(decoded_latent_uni['act'].detach().cpu().numpy() * action_stdv + action_mean, -1, 1)
    x_disp_full = np.append(x_disp_full, decoded_act[:, 0])
    y_disp_full = np.append(y_disp_full, decoded_act[:, 1])
    z_disp_full = np.append(z_disp_full, decoded_act[:, 2])
    gripper_full = np.append(gripper_full, decoded_act[:, 3])
print(len(x_disp_full))

ref_labels = np.load('/mnt/datashare/yelhelw/combin_tasks/actions_val.npy', mmap_mode="r")
ref_labels_norm = np.clip(ref_labels.copy(), -1, 1)
x_disp_ref = np.append(x_disp, ref_labels_norm[:, 0])
y_disp_ref = np.append(y_disp, ref_labels_norm[:, 1])
z_disp_ref = np.append(z_disp, ref_labels_norm[:, 2])
gripper_ref = np.append(gripper, ref_labels_norm[:, 3])

[0.02619656 0.02819326 0.02302605 ... 0.02669895 0.07254413 0.07252748]
Transforming latent vectors...
60055
100085


In [42]:
#Embeddings per modality
embedding_mod_gw = {}

for m in modalities:
    embedding = reducer.transform(latents_dict[m][:])
    #embedding_combin = reducer.transform(latents_dict_combin[m])
    #embedding_mod_gw[m] = np.concatenate((embedding, embedding_combin))
    embedding_mod_gw[m] = embedding
    


In [48]:
print(len(attributes[:,0]))
floor_color = attributes[:,-4]
wall_color = attributes[:,-3]
ball_color = attributes[:,-2]
wall_binary = np.where((wall==-10),0,1)
goal_binary = np.where((goal==-10),0,1)
ball_binary = np.where((ball<0),0,1)

# Create obj_code with different combinations
obj_code = np.zeros(len(wall_binary), dtype=object)
obj_code[:] = "none"

# all - wall AND goal AND ball
mask_all = (wall_binary == 1) & (goal_binary == 1) & (ball_binary == 1)
obj_code[mask_all] = "all"

# wall_ball - wall AND ball (goal False)
mask_wall_ball = (wall_binary == 1) & (ball_binary == 1) & (goal_binary == 0)
obj_code[mask_wall_ball] = "wall_ball"

# wall - only wall
mask_wall_only = (wall_binary == 1) & (goal_binary == 0) & (ball_binary == 0)
obj_code[mask_wall_only] = "wall"

# goal_ball - goal AND ball (wall False)
mask_goal_ball = (goal_binary == 1) & (ball_binary == 1) & (wall_binary == 0)
print(np.where(mask_goal_ball==True))
obj_code[mask_goal_ball] = "goal_ball"

100085
(array([ 40036,  40037,  40038, ..., 100072, 100074, 100080]),)


In [49]:
embedding_mod = embedding_mod_gw

modalities = ['v']
modality_cmap = ListedColormap(['red', 'green', 'orange'])

all_embeddings = [embedding_mod] + list(embedding_mod.values())
emb = embedding_mod['v']
x_min = min([emb[:, 0].min()])
x_max = max([emb[:, 0].max() ])
y_min = min([emb[:, 1].min() ])
y_max = max([emb[:, 1].max() ])

x_range = x_max - x_min
y_range = y_max - y_min
padding = 0.05
x_limits = [x_min - padding * x_range, x_max + padding * x_range]
y_limits = [y_min - padding * y_range, y_max + padding * y_range]

actions = ['right-left', 'front-back', 'up-down', 'gripper']
n_actions = len(actions)
n_modalities = len(modalities)

n_cols = n_modalities 
n_rows = n_actions 

fig, axes = plt.subplots(n_rows, n_cols + 1, figsize=(6 * n_cols + 2, 4 * n_rows))
plt.subplots_adjust(hspace=0.3, wspace=0.3, right=0.92)

###################################
# Top row: Modality plot centered #
###################################
#axes[0, 0].remove()
#axes[0, 2].remove()
index = 0
for modality in modalities:
    # GW embeddings
    modality_ax = axes[0, modalities.index(modality)]
    modality_ax.set_title(f'UMAP - {modality} in GW', fontsize=16, fontweight='bold')
    scatter = modality_ax.scatter(
        embedding_mod_gw[modality][:, 0],
        embedding_mod_gw[modality][:, 1],
        s=5,
    )
    modality_ax.set_xlim(x_limits)
    modality_ax.set_ylim(y_limits)

modality_legend_elements = []
for i, color in enumerate(modality_cmap.colors):
    modality_legend_elements.append(plt.Line2D([0], [0], marker='o', color='w', 
                                    markerfacecolor=color, markersize=8, 
                                    label=f'{modality_names[i]}'))

axes[0, -1].axis('off')
#axes[0, -1].legend(handles=modality_legend_elements, title='Modality', 
 #                   loc='center left', fontsize=10)
plt.suptitle('UMAP Visualization - Modalities and Attributes', fontsize=20, fontweight='bold', y=0.98)
plt.savefig(f"graphs/UMAP_feb.png")


In [50]:
actions = ['right-left', 'front-back',  'gripper','objects']
n_actions = len(actions)
n_modalities = len(modalities)

###################################
# Randomize plotting order to avoid points covering each other
###################################
np.random.seed(42)
shuffle_idx = np.random.permutation(len(x_disp))
wall_shuf = wall
goal_shuf = goal
ball_shuf = ball
# Create shuffled versions of all data
x_disp_shuf = x_disp[shuffle_idx]
#x_disp_ref_shuf = x_disp_ref[shuffle_idx]
y_disp_shuf = y_disp[shuffle_idx]
#y_disp_ref_shuf = y_disp_ref[shuffle_idx]
z_disp_shuf = z_disp[shuffle_idx]
#z_disp_ref_shuf = z_disp_ref[shuffle_idx]
gripper_shuf = gripper[shuffle_idx]
#gripper_ref_shuf = gripper_ref[shuffle_idx]
modalities = ['v']

color_map = {
    #"none": '#1f77b4',
    "all": '#ff7f0e',
    "wall_ball": '#2ca02c',
    "wall": '#d62728',
    "goal_ball": '#9467bd'
}

# Shuffle embeddings
#embedding_v_shuf = embedding_v[shuffle_idx]
#embedding_act_shuf = embedding_act[shuffle_idx]
###################################
# Define masks on shuffled data
###################################
x_mouvement = ((x_disp_shuf==1)|(x_disp_shuf==-1))
x_stop = (np.abs(x_disp_shuf)<0.1)

x_disp_binary = x_disp_shuf[x_mouvement]
right_left_labels = np.where(x_disp_binary<0, 1,0)

y_filter = (y_disp_shuf>-10)
y_filtered= y_disp_shuf
front_back_labels = np.where((np.abs(y_filtered)<0.2), 0,1)

wall_binary = np.where((wall_shuf==-10),0,1)
goal_binary = np.where((goal_shuf==-10),0,1)
ball_binary = np.where((ball_shuf<0),0,1)
binary_map = ListedColormap(['red', 'green'])

for row, action in enumerate(actions):
    actual_row = row 
    for modality in modalities:
        ax = axes[actual_row, modalities.index(modality)]
        embedding =embedding_mod[modality][shuffle_idx]
        if action == 'right-left':
            ax.scatter(
                embedding_mod[modality][:, 0],
                embedding_mod[modality][:, 1],
                c='grey',
                s=5,
                alpha=0.3
            )
            scatter = ax.scatter(
                embedding[:, 0],
                embedding[:, 1],
                c=x_disp_shuf,
                cmap='plasma',
                s=5,
                alpha=0.9
            )
            ax.set_title(f' Right-Left')

            axes[actual_row, -1].axis('off')
            pos = axes[actual_row, -1].get_position()
            cbar_height = max(0.02, pos.height * 0.6) 
            cbar_y = pos.y0 + (pos.height - cbar_height) / 2 
            cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
            cbar = plt.colorbar(scatter, cax=cbar_ax)
            
        elif action == 'front-back':
            ax.scatter(
                embedding_mod[modality][:, 0],
                embedding_mod[modality][:, 1],
                c='grey',
                s=5,
                alpha=0.3
            )
            scatter = ax.scatter(
                embedding[y_filter, 0],
                embedding[y_filter, 1],
                c=front_back_labels,
                cmap=binary_map,
                s=5,
                alpha=0.9
            )
            ax.set_title(f'Front-Stop')

            axes[actual_row, -1].axis('off')
            pos = axes[actual_row, -1].get_position()
            cbar_height = max(0.02, pos.height * 0.6) 
            cbar_y = pos.y0 + (pos.height - cbar_height) / 2
            cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
            cbar = plt.colorbar(scatter, cax=cbar_ax)
        elif action == 'gripper':
            ax.scatter(
                embedding_mod[modality][:, 0],
                embedding_mod[modality][:, 1],
                c='grey',
                s=5,
                alpha=0.3
            )
            scatter = ax.scatter(
                embedding[:, 0],
                embedding[:, 1],
                c=gripper_shuf,
                cmap='plasma',
                s=5,
                alpha=0.7
            )
            ax.set_title(f'Gripper')
            

            axes[actual_row, -1].axis('off')
            pos = axes[actual_row, -1].get_position()
            cbar_height = max(0.02, pos.height * 0.6) 
            cbar_y = pos.y0 + (pos.height - cbar_height) / 2
            cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
            cbar = plt.colorbar(scatter, cax=cbar_ax)
        elif action == 'goal':
            scatter = ax.scatter(
                embedding_mod_gw[modality][:, 0],
                embedding_mod_gw[modality][:, 1],
                c=goal_binary[:],
                cmap=binary_map,
                s=5,
                alpha=0.9
            )
            ax.set_title(f'Goal')

            axes[actual_row, -1].axis('off')
            pos = axes[actual_row, -1].get_position()
            cbar_height = max(0.02, pos.height * 0.6) 
            cbar_y = pos.y0 + (pos.height - cbar_height) / 2
            cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
            cbar = plt.colorbar(scatter, cax=cbar_ax)
        elif action == 'ball':
            scatter = ax.scatter(
                embedding_mod_gw[modality][:60019, 0],
                embedding_mod_gw[modality][:60019, 1],
                c=ball_binary[:60019],
                cmap=binary_map,
                s=5,
                alpha=0.9
            )
            ax.set_title(f'Ball')

            axes[actual_row, -1].axis('off')
            pos = axes[actual_row, -1].get_position()
            cbar_height = max(0.02, pos.height * 0.6) 
            cbar_y = pos.y0 + (pos.height - cbar_height) / 2
            cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
            cbar = plt.colorbar(scatter, cax=cbar_ax)
        elif action == 'wall':
            scatter = ax.scatter(
                embedding_mod_gw[modality][:60019, 0],
                embedding_mod_gw[modality][:60019, 1],
                c=wall_binary[:60019],
                cmap=binary_map,
                s=5,
                alpha=0.9
            )
            ax.set_title(f'Wall')

            axes[actual_row, -1].axis('off')
            pos = axes[actual_row, -1].get_position()
            cbar_height = max(0.02, pos.height * 0.6) 
            cbar_y = pos.y0 + (pos.height - cbar_height) / 2
            cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
            cbar = plt.colorbar(scatter, cax=cbar_ax)
        
        elif action == 'objects':
            for obj_label, color in color_map.items():
                mask = obj_code == obj_label
                ax.scatter(embedding_mod_gw[modality][mask, 0],
                embedding_mod_gw[modality][mask, 1],
                c=color,
                alpha=0.8,
                s=5,
                label=obj_label,
                )
            ax.set_title(f'Objects')

            axes[actual_row, -1].axis('off')
            pos = axes[actual_row, -1].get_position()
            handles = [plt.Line2D([0], [0], marker='o', color='w', markerfacecolor=color, markersize=8, label=obj_label)
                       for obj_label, color in color_map.items()]
            axes[actual_row, -1].legend(handles=handles, loc='center left', fontsize=9)
            
            
        ax.set_xlim(x_limits)
        ax.set_ylim(y_limits)

plt.suptitle('UMAP Visualization - Modalities and Attributes', fontsize=20, fontweight='bold', y=0.98)
plt.savefig(f"graphs/UMAP_feb.png")
plt.close()

In [ ]:
# =============================================================================
# CONFIGURATION DICTIONARY - Edit masks here to control visualizations
# Based on histogram analysis of action distributions
# =============================================================================

# Each action row has: 'masks' (list of filter conditions), 'colors', 'labels', 'title'
# Use None for continuous colormap instead of discrete categories

# Define masks based on histogram peaks and action modes
action_config = {
    'Movement Direction': {
        'type': 'categorical',
        'categories': [
            #{'mask': (np.abs(x_disp) < 0.3) & (y_disp > 0.5), 'color': 'green', 'label': 'Push Forward'},
            #{'mask': (x_disp < -0.5) & (np.abs(y_disp) < 0.3), 'color': 'blue', 'label': 'Push Left'},
            #{'mask': (x_disp > 0.5) & (np.abs(y_disp) < 0.3), 'color': 'red', 'label': 'Push Right'},
            {'mask': (np.abs(x_disp) > 0.1) & (np.abs(y_disp) < 0.2), 'color': 'orange', 'label': 'Stationary'},
        ],
        'show_background': True,
        'title': 'Movement Direction (XY plane)',
    },
    'Diagonal Movements': {
        'type': 'categorical',
        'categories': [
            {'mask': (x_disp < -0.1) , 'color': 'purple', 'label': 'Diagonal Left-Forward'},
            {'mask': (x_disp > 0.), 'color': 'cyan', 'label': 'Diagonal Right-Forward'},
            #{'mask': (x_disp < -0.3) & (y_disp < -0.3), 'color': 'brown', 'label': 'Diagonal Left-Back'},
            #{'mask': (x_disp > 0.3) & (y_disp < -0.3), 'color': 'pink', 'label': 'Diagonal Right-Back'},
        ],
        'show_background': True,
        'title': 'Diagonal Movements',
    },
    'Vertical + Gripper': {
        'type': 'categorical',
        'categories': [
            #{'mask': (z_disp > 0.5) & (gripper > 0.5), 'color': 'green', 'label': 'Up + Grip Close'},
            #{'mask': (z_disp > 0.5) & (gripper < -0.2), 'color': 'blue', 'label': 'Up + Grip Open'},
            {'mask': (z_disp < -0.5) & (gripper > 0.5), 'color': 'red', 'label': 'Down + Grip Close'},
            #{'mask': (z_disp < -0.5) & (gripper < -0.2), 'color': 'orange', 'label': 'Down + Grip Open'},
        ],
        'show_background': True,
        'title': 'Vertical Movement + Gripper',
    },
    'X-Displacement': {
        'type': 'continuous',
        'mask': None,
        'color_by': x_disp,
        'cmap': 'coolwarm',
        'title': 'X Displacement (Left-Right)',
    },
    'Y-Displacement': {
        'type': 'continuous',
        'mask': None,
        'color_by': y_disp,
        'cmap': 'coolwarm',
        'title': 'Y Displacement (Front-Back)',
    },
    'Gripper State': {
        'type': 'continuous',
        'mask': None,
        'color_by': gripper,
        'cmap': 'plasma',
        'title': 'Gripper Opening',
    },
}

# =============================================================================
# PLOTTING CODE
# =============================================================================

n_actions = len(action_config)
n_modalities = len(modalities)
n_rows = n_actions + 1
n_cols = n_modalities
modalities = ['v', 'act']
fig, axes = plt.subplots(n_rows, n_cols + 1, figsize=(6 * n_cols + 2, 4 * n_rows))
plt.subplots_adjust(hspace=0.3, wspace=0.3, right=0.92)

# Top row: plain modality embeddings
for col, modality in enumerate(modalities):
    ax = axes[0, col]
    ax.set_title(f'UMAP - {modality.upper()} in GW', fontsize=14, fontweight='bold')
    ax.scatter(embedding_mod_gw[modality][:, 0], embedding_mod_gw[modality][:, 1], s=5, c='steelblue', alpha=0.5)
    ax.set_xlim(x_limits)
    ax.set_ylim(y_limits)
axes[0, -1].axis('off')

# Action rows
for row, (action_name, config) in enumerate(action_config.items()):
    actual_row = row + 1
    
    for col, modality in enumerate(modalities):
        ax = axes[actual_row, col]
        embedding = embedding_mod_gw[modality]
        
        if config['type'] == 'continuous':
            # Continuous colormap
            mask = config.get('mask')
            if mask is None:
                scatter = ax.scatter(embedding[:, 0], embedding[:, 1],
                                     c=config['color_by'], cmap=config['cmap'], s=5, alpha=0.8)
            else:
                scatter = ax.scatter(embedding[mask, 0], embedding[mask, 1],
                                     c=config['color_by'], cmap=config['cmap'], s=5, alpha=0.8)
            
            # Add colorbar on last column
            if col == n_modalities - 1:
                axes[actual_row, -1].axis('off')
                pos = axes[actual_row, -1].get_position()
                cbar_height = max(0.02, pos.height * 0.6)
                cbar_y = pos.y0 + (pos.height - cbar_height) / 2
                cbar_ax = fig.add_axes([pos.x0 + 0.02, cbar_y, 0.02, cbar_height])
                plt.colorbar(scatter, cax=cbar_ax)
                
        elif config['type'] == 'categorical':
            # Categorical with multiple masks
            if config.get('show_background', True):
                ax.scatter(embedding[:, 0], embedding[:, 1], c='lightgrey', s=1, alpha=0.3)
            
            for cat in config['categories']:
                mask = cat['mask']
                ax.scatter(embedding[mask, 0], embedding[mask, 1],
                           c=cat['color'], s=5, alpha=0.8, label=cat['label'])
            
            # Add legend on last column
            if col == n_modalities - 1:
                axes[actual_row, -1].axis('off')
                handles = [plt.Line2D([0], [0], marker='o', color='w', 
                           markerfacecolor=cat['color'], markersize=8, label=cat['label'])
                           for cat in config['categories']]
                axes[actual_row, -1].legend(handles=handles, loc='center left', fontsize=9)
        
        ax.set_title(f"{modality.upper()} - {config['title']}", fontsize=10)
        ax.set_xlim(x_limits)
        ax.set_ylim(y_limits)

plt.suptitle('UMAP: Action Dimensions Across Modalities', fontsize=18, fontweight='bold', y=0.98)
plt.savefig(f"graphs/{current_model}/UMAP_action_config_comparison.png", dpi=150)
plt.show()

# Print mask statistics
print("\n=== Mask Statistics ===")
for action_name, config in action_config.items():
    print(f"\n{action_name}:")
    if config['type'] == 'categorical':
        for cat in config['categories']:
            print(f"  {cat['label']}: {cat['mask'].sum()} samples ({100*cat['mask'].sum()/len(x_disp):.1f}%)")
    elif config.get('mask') is not None:
        print(f"  Filtered: {config['mask'].sum()} samples ({100*config['mask'].sum()/len(x_disp):.1f}%)")

In [ ]:
# Interactive downloadable figure using Plotly
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Prepare data for the three subplots
embedding = embedding_mod_gw['v'][shuffle_idx]

# Create figure with 3 subplots
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=('Right-Left (X displacement)', 'Front-Back (Y displacement)', 'Gripper State'),
    specs=[[{'type': 'scatter'}, {'type': 'scatter'}, {'type': 'scatter'}]]
)

# Right-Left subplot
fig.add_trace(
    go.Scatter(
        x=embedding[:, 0],
        y=embedding[:, 1],
        mode='markers',
        marker=dict(
            size=5,
            color=x_disp_shuf,
            colorscale='Plasma',
            showscale=True,
            colorbar=dict(x=0.32, len=0.4, title="X Disp"),
            opacity=0.6
        ),
        text=[f"X: {x:.3f}<br>Y: {y:.3f}<br>Z: {z:.3f}<br>Grip: {g:.3f}<br>Emb: ({e[0]:.2f}, {e[1]:.2f})" 
              for x, y, z, g, e in zip(x_disp[shuffle_idx], y_disp[shuffle_idx], z_disp[shuffle_idx], 
                                       gripper[shuffle_idx], embedding)],
        hovertemplate='%{text}<extra></extra>',
        name='Right-Left'
    ),
    row=1, col=1
)

# Add background points
fig.add_trace(
    go.Scatter(
        x=embedding_mod_gw['v'][:, 0],
        y=embedding_mod_gw['v'][:, 1],
        mode='markers',
        marker=dict(size=1, color='lightgray', opacity=0.2),
        hoverinfo='skip',
        showlegend=False
    ),
    row=1, col=1
)

# Front-Back subplot (only points where y_filter is True)
filtered_emb_idx = np.where(y_filter[shuffle_idx])[0]
fig.add_trace(
    go.Scatter(
        x=embedding[filtered_emb_idx, 0],
        y=embedding[filtered_emb_idx, 1],
        mode='markers',
        marker=dict(
            size=5,
            color=front_back_labels,
            colorscale='RdYlGn',
            showscale=True,
            colorbar=dict(x=0.98, len=0.4, title="Front/Back"),
            opacity=0.6
        ),
        text=[f"X: {x:.3f}<br>Y: {y:.3f}<br>Z: {z:.3f}<br>Grip: {g:.3f}<br>Emb: ({e[0]:.2f}, {e[1]:.2f})" 
              for x, y, z, g, e in zip(x_disp[shuffle_idx[filtered_emb_idx]], 
                                       y_disp[shuffle_idx[filtered_emb_idx]], 
                                       z_disp[shuffle_idx[filtered_emb_idx]], 
                                       gripper[shuffle_idx[filtered_emb_idx]], 
                                       embedding[filtered_emb_idx])],
        hovertemplate='%{text}<extra></extra>',
        name='Front-Back'
    ),
    row=1, col=2
)

# Add background points
fig.add_trace(
    go.Scatter(
        x=embedding_mod_gw['v'][:, 0],
        y=embedding_mod_gw['v'][:, 1],
        mode='markers',
        marker=dict(size=1, color='lightgray', opacity=0.2),
        hoverinfo='skip',
        showlegend=False
    ),
    row=1, col=2
)

# Gripper subplot
fig.add_trace(
    go.Scatter(
        x=embedding[:, 0],
        y=embedding[:, 1],
        mode='markers',
        marker=dict(
            size=5,
            color=gripper_shuf,
            colorscale='Plasma',
            showscale=True,
            colorbar=dict(x=1.64, len=0.4, title="Gripper"),
            opacity=0.6
        ),
        text=[f"X: {x:.3f}<br>Y: {y:.3f}<br>Z: {z:.3f}<br>Grip: {g:.3f}<br>Emb: ({e[0]:.2f}, {e[1]:.2f})" 
              for x, y, z, g, e in zip(x_disp[shuffle_idx], y_disp[shuffle_idx], z_disp[shuffle_idx], 
                                       gripper[shuffle_idx], embedding)],
        hovertemplate='%{text}<extra></extra>',
        name='Gripper'
    ),
    row=1, col=3
)

# Add background points
fig.add_trace(
    go.Scatter(
        x=embedding_mod_gw['v'][:, 0],
        y=embedding_mod_gw['v'][:, 1],
        mode='markers',
        marker=dict(size=1, color='lightgray', opacity=0.2),
        hoverinfo='skip',
        showlegend=False
    ),
    row=1, col=3
)

# Update layout
fig.update_xaxes(title_text="UMAP 1", row=1, col=1)
fig.update_yaxes(title_text="UMAP 2", row=1, col=1)
fig.update_xaxes(title_text="UMAP 1", row=1, col=2)
fig.update_yaxes(title_text="UMAP 2", row=1, col=2)
fig.update_xaxes(title_text="UMAP 1", row=1, col=3)
fig.update_yaxes(title_text="UMAP 2", row=1, col=3)

fig.update_layout(
    title_text='Interactive Action Visualization - Hover to see action values',
    height=600,
    width=2000,
    hovermode='closest',
    showlegend=False
)

# Save as HTML
output_path = f'graphs/interactive_action_visualization.html'
fig.write_html(output_path)
print(f"Interactive visualization saved to {output_path}")
print(f"Download and open in a web browser to interact with the figure")